# 03 · Evaluación y comparación

Lee los runs del MLflow configurado en `MLFLOW_TRACKING_URI` (por defecto, el de la aplicación).

In [ ]:
import mlflow
import pandas as pd

from potato_blight.compare import compare
from potato_blight.mlflow_utils import setup

setup("papasana-ap01")
exp = mlflow.get_experiment_by_name("papasana-ap01")
folds = mlflow.search_runs([exp.experiment_id], "tags.run_type = 'cv_fold' and tags.smoke_test = 'false'")
cols = ["tags.architecture", "tags.fold", "metrics.test_macro_f1", "metrics.test_mcc", "metrics.test_macro_auc_roc",
        "metrics.test_ece"]
folds[[c for c in cols if c in folds]].sort_values(["tags.architecture", "tags.fold"])

In [ ]:
# Media ± desviación por arquitectura y escenario externo
ext = [c for c in folds.columns if c.startswith("metrics.external_") and c.endswith("_macro_f1")]
folds.groupby("tags.architecture")[["metrics.test_macro_f1", *ext]].agg(["mean", "std"])

In [ ]:
resultado = compare("papasana-ap01", metric="macro_f1")
pd.DataFrame(resultado.get("wilcoxon_holm", []))

In [ ]:
resultado.get("friedman"), pd.DataFrame(resultado.get("mcnemar", [])), pd.DataFrame(resultado.get("delong", []))